In [1]:
import numpy as np
import pandas as pd
import yaml
from pathlib import Path

# Optional (only needed to read 3D checkpoint centroids)
try:
    import torch
    HAVE_TORCH = True
except Exception:
    HAVE_TORCH = False

root = Path("/Users/gherardi/Documents/GitHub/glider_optimization_debug")
cfg_path = root / "conf/test.yaml"
cfg = yaml.safe_load(cfg_path.read_text()) or {}

nf = cfg.get("neuralFoilSampling", {})

# === Constants copied from glider_jinenv.initDyn() ===
m = 0.065
l_w_i = -0.005
l_w_f = -0.015
l = 0.26
l_e = 0.02
S_w = 0.158
S_e = 0.017

# Same helper as glider_jinenv.mc_to_wcom(l_w): return l_w + 0.003
def mc_to_wcom(l_w):
    return l_w + 0.003

# Shared derived values
m_w = 0.6 * m * S_w / (S_w + S_e)
m_e = 0.6 * m * S_e / (S_w + S_e)
m_f = 0.4 * m
l_w = 0.5 * (l_w_i + l_w_f)
l_w_m = (l_w_i + l_w_f) / 2
com_w_2d = l_w_m + mc_to_wcom(l_w_m)   # exactly as code
com_e_2d = l + l_e
l_f = -(l_w * m_w + (l - l_e) * m_e) / m_f
print("Derived fuselage length l_f:", l_f)
com_f = l_f

# 2D aerodynamic reference centroid (scalar x)
com_a_2d = (com_w_2d * m_w + com_e_2d * m_e + com_f * m_f) / (m_w + m_e + m_f)

# 3D defaults (before checkpoint override)
com_w_x = float(com_w_2d)
com_w_z = 0.0
com_e_x = float(com_e_2d)
com_e_z = 0.0

# 3D checkpoint override (same logic as code)
use_3d_llt = bool(nf.get("use_3d_llt", False))
ckpt_rel = nf.get("llt_ckpt_path", "")
ckpt_path = (root / ckpt_rel).resolve() if ckpt_rel else None

centroid_src = "defaults (no checkpoint override)"
if use_3d_llt and HAVE_TORCH and ckpt_path and ckpt_path.exists():
    ckpt = torch.load(str(ckpt_path), map_location="cpu")
    cent = ckpt.get("centroid", {}) if isinstance(ckpt, dict) else {}
    if "wing_x" in cent: com_w_x = float(cent["wing_x"])
    if "wing_z" in cent: com_w_z = float(cent["wing_z"])
    if "elevator_x" in cent: com_e_x = float(cent["elevator_x"])
    if "elevator_z" in cent: com_e_z = float(cent["elevator_z"])
    centroid_src = f"checkpoint: {ckpt_path}"
elif use_3d_llt and not HAVE_TORCH:
    centroid_src = "defaults (torch not available)"
elif use_3d_llt and ckpt_path and not ckpt_path.exists():
    centroid_src = f"defaults (checkpoint missing: {ckpt_path})"

# 3D aerodynamic reference centroid used by dynamics (x only; code assumes com_a_z=0)
com_a_x_3d = (com_w_x * m_w + com_e_x * m_e + com_f * m_f) / (m_w + m_e + m_f)
com_a_z_3d_assumed = 0.0

# Body-frame lever arms used later in 3D torque path
r_w_bx_3d = -com_w_x + com_a_x_3d
r_w_bz_3d = -com_w_z
r_e_bx_3d = -com_e_x + com_a_x_3d
r_e_bz_3d = -com_e_z

# 2D equivalent lever scalar (before world rotation)
r_w_2d_scalar = -com_w_2d + com_a_2d
r_e_2d_scalar = -com_e_2d + com_a_2d

print("Centroid source for 3D:", centroid_src)
print("use_3d_llt:", use_3d_llt)

summary = pd.DataFrame([
    {"mode":"2D", "com_w_x":com_w_2d, "com_w_z":0.0, "com_e_x":com_e_2d, "com_e_z":0.0, "com_a_x":com_a_2d, "com_a_z":0.0},
    {"mode":"3D", "com_w_x":com_w_x,  "com_w_z":com_w_z, "com_e_x":com_e_x, "com_e_z":com_e_z, "com_a_x":com_a_x_3d, "com_a_z":com_a_z_3d_assumed},
])
display(summary)

lever_cmp = pd.DataFrame([
    {"mode":"2D", "r_w_bx_or_scalar":r_w_2d_scalar, "r_w_bz":0.0,      "r_e_bx_or_scalar":r_e_2d_scalar, "r_e_bz":0.0},
    {"mode":"3D", "r_w_bx_or_scalar":r_w_bx_3d,     "r_w_bz":r_w_bz_3d,"r_e_bx_or_scalar":r_e_bx_3d,     "r_e_bz":r_e_bz_3d},
])
display(lever_cmp)

delta = summary.iloc[1][["com_w_x","com_w_z","com_e_x","com_e_z","com_a_x","com_a_z"]] - \
        summary.iloc[0][["com_w_x","com_w_z","com_e_x","com_e_z","com_a_x","com_a_z"]]
print("3D - 2D centroid deltas:")
display(delta.to_frame("delta").T)

Derived fuselage length l_f: -0.021428571428571436
Centroid source for 3D: checkpoint: /Users/gherardi/Documents/GitHub/glider_optimization_debug/artifacts/models/3d_blocks.pt
use_3d_llt: True


,mode,com_w_x,com_w_z,com_e_x,com_e_z,com_a_x,com_a_z
0,2D,-0.017000,0.000000,0.280000,0.000000e+00,-0.001461,0.0
1,3D,0.097125,0.020381,0.025227,2.110199e-19,0.045513,0.0


,mode,r_w_bx_or_scalar,r_w_bz,r_e_bx_or_scalar,r_e_bz
0,2D,0.015539,0.000000,-0.281461,0.000000e+00
1,3D,-0.051612,-0.020381,0.020286,-2.110199e-19


3D - 2D centroid deltas:


,com_w_x,com_w_z,com_e_x,com_e_z,com_a_x,com_a_z
delta,0.114125,0.020381,-0.254773,0.0,0.046973,0.0


In [2]:
from pathlib import Path
import pprint
import torch

pt_path = Path("/Users/gherardi/Documents/GitHub/glider_optimization_debug/artifacts/models/3d_blocks.pt")
print("Path:", pt_path)
print("Exists:", pt_path.exists())

obj = torch.load(str(pt_path), map_location="cpu")
print("Top-level type:", type(obj))

if isinstance(obj, dict):
    print("\nTop-level keys:")
    for k in sorted(obj.keys()):
        v = obj[k]
        shape = tuple(v.shape) if hasattr(v, "shape") else None
        print(f"- {k}: type={type(v).__name__}, shape={shape}")

    # Print centroid/details if present
    for special in ["centroid", "meta", "config", "state_dict"]:
        if special in obj:
            print(f"\n{special}:")
            if isinstance(obj[special], dict):
                pprint.pprint({kk: obj[special][kk] for kk in list(obj[special].keys())[:30]})
            else:
                print(obj[special])
else:
    print("Object preview:")
    print(obj)

Path: /Users/gherardi/Documents/GitHub/glider_optimization_debug/artifacts/models/3d_blocks.pt
Exists: True
Top-level type: <class 'dict'>

Top-level keys:
- beta: type=float, shape=None
- centroid: type=dict, shape=None
- config_path: type=str, shape=None
- device: type=str, shape=None
- elevator_geometry: type=dict, shape=None
- elevator_requires_grad: type=bool, shape=None
- enforce_symmetry: type=bool, shape=None
- flow: type=dict, shape=None
- model_size: type=str, shape=None
- n_iter: type=int, shape=None
- tol: type=float, shape=None
- wing_geometry: type=dict, shape=None
- wing_requires_grad: type=bool, shape=None

centroid:
{'elevator_x': 0.02522654693991627,
 'elevator_z': 2.1101987955775332e-19,
 'wing_x': 0.09712500253101201,
 'wing_z': 0.02038103043897718}


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

root = Path("/Users/gherardi/Documents/GitHub/glider_optimization_debug/diagnostics/2026-02-13_3d-llt-debug")
f2 = root / "rollout_terms_stage_0_16_full_test2a_3dwing_fresh.csv"
f3 = root / "rollout_terms_stage_0_16_full_test5_3dwing_3delev_centroidLplusX_20260219_170659.csv"

def check(file):
    df = pd.read_csv(file).copy()
    # two candidate conventions
    df["tau_from_plus"] = df["r_e_x"]*df["F_ez"] - df["r_e_z"]*df["F_ex"] + df["M_e"]
    df["tau_from_minus"] = -df["r_e_x"]*df["F_ez"] + df["r_e_z"]*df["F_ex"] + df["M_e"]
    df["err_plus"] = df["tau_e"] - df["tau_from_plus"]
    df["err_minus"] = df["tau_e"] - df["tau_from_minus"]

    print("\nFILE:", file.name)
    print("mean |tau_e - tau_from_plus| :", np.nanmean(np.abs(df["err_plus"])))
    print("mean |tau_e - tau_from_minus|:", np.nanmean(np.abs(df["err_minus"])))
    display(df.loc[:3, [
        "stage","r_e_x","r_e_z","F_ex","F_ez","M_e",
        "tau_e_term_rxfz","tau_e_term_rzfx","tau_e",
        "tau_from_plus","tau_from_minus"
    ]])

check(f2)
check(f3)


FILE: rollout_terms_stage_0_16_full_test2a_3dwing_fresh.csv
mean |tau_e - tau_from_plus| : 0.050334770708270436
mean |tau_e - tau_from_minus|: 9.122604632489797e-17


,stage,r_e_x,r_e_z,F_ex,F_ez,M_e,tau_e_term_rxfz,tau_e_term_rzfx,tau_e,tau_from_plus,tau_from_minus
0,0,-0.281461,-0.000000,2.793966e-08,-0.419095,0.000937,-0.117959,-0.000000,-0.117022,0.118896,-0.117022
1,1,-0.281461,-0.000000,1.035532e-08,-0.231912,0.000556,-0.065274,-0.000000,-0.064718,0.065831,-0.064718
2,2,-0.281407,-0.005468,2.263800e-03,-0.116506,0.000288,-0.032786,-0.000012,-0.032510,0.033086,-0.032510
3,3,-0.281137,-0.013496,1.869643e-03,-0.038948,0.000097,-0.010950,-0.000025,-0.010878,0.011072,-0.010878



FILE: rollout_terms_stage_0_16_full_test5_3dwing_3delev_centroidLplusX_20260219_170659.csv
mean |tau_e - tau_from_plus| : 5.953584315071298e+238
mean |tau_e - tau_from_minus|: 1.1652569176435556e+254


,stage,r_e_x,r_e_z,F_ex,F_ez,M_e,tau_e_term_rxfz,tau_e_term_rzfx,tau_e,tau_from_plus,tau_from_minus
0,0,-0.224559,-2.110199e-19,0.046321,-0.442899,0.001042,0.099457,9.774662e-21,0.100499,0.100499,-0.098415
1,1,-0.224559,-2.110199e-19,0.001246,-0.726024,0.001447,0.163036,2.629258e-22,0.164482,0.164482,-0.161589
2,2,-0.223927,1.683921e-02,-0.105938,-1.408763,0.002243,0.315460,1.783910e-03,0.319487,0.319487,-0.315001
3,3,-0.217780,5.475958e-02,-0.822619,-3.271579,0.003602,0.712486,4.504626e-02,0.761134,0.761134,-0.753930
